In [1]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 0 modules, rebound 0 names.


In [2]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, scaled_quantile_loss

alt.renderers.enable("jupyter", offline=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [4]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import _style, plot_sample
from ts_uncertainty.synthetic import SyntheticGenerator
from ts_uncertainty.real import RealSampler

# data settings
context_length = 256
horizon_length = 128
samples_batch = 100
seed = 42

## Synthetic Samples

In [ ]:
sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    # n_periods=1,
    n_harmonics=3,             # 1 harmonic = pure sine, no extra shape
    # level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    # amplitude_range=(1.0, 1.0),
    seed=seed
)

# generate samples
samples_synthetic = sin_gen.sample_batch(samples_batch)

In [8]:
sample_to_plot = 0
plot_sample(samples_synthetic[sample_to_plot], title=f"Synthetic Sample {sample_to_plot}")

## Chaotic Samples

In [10]:
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=None,                    # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)

# generate samples
samples_chaotic = chaos_gen.sample_batch(samples_batch)

In [11]:
sample_to_plot = 0
plot_sample(samples_chaotic[sample_to_plot], title=f"Chaotic Sample {sample_to_plot}")

## Real Sample
Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [12]:
data_path_test = "../data/test.csv"
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series = len(data_test['series_index'].unique())
print(f"Number of time series in the test set: {number_series}")
data_test.head()

Number of time series in the test set: 361


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [16]:
real_gen = RealSampler(
    df=data_test,
    context_length=context_length,
    horizon=horizon_length,
    normalize='none', # dont normalize
    seed=seed,
)

# generate samples
samples_real = real_gen.sample_batch(samples_batch)

In [18]:
sample_to_plot = 0
plot_sample(samples_real[sample_to_plot], title=f"Real Sample {sample_to_plot}")

# Models

In [20]:
models = {
    "AutoARIMA": {'name': 'autoarima'},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "Timestral (123M)": {'name': 'timestral', 'size': '123M'},
}

# prepare and download all models
for key, m in models.items():
    print(f"Preparing model: {key}")
    model = ForecastModel(**m)

Preparing model: AutoARIMA
Preparing model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preparing model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: Timestral (123M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

# Experiments

In [ ]:




inj = Injector(
    noise=False,
    shift=True,
    shift_covariate=True,
    shift_recurring_prob=0.8,
    shift_duration_range=(0.1, 0.25),
    seed=seed,
)
s = inj(sin_gen.sample())
plot_sample(s, title="Sin Sample")